# Dia7 - Spark Processing Examples (PySpark DataFrame API)

**Objetivo:** Demonstrar operações da PySpark DataFrame API (filter, groupBy, agg, join, lazy evaluation, explain) sobre as tabelas do projeto Indian E-Commerce.

| Item | Valor |
|---|---|
| **Camada** | Leitura de Bronze/Gold + transformações em memória |
| **Entrada** | `{catalog}.gold.fat_vendas`, `{catalog}.bronze.{sales,customers,products}` |
| **Saída** | `{catalog}.gold.daily_revenue`, `{catalog}.gold.revenue_by_category` |
| **Ordem no pipeline** | Após `Dia6a-Gerador-CSVs-Indian` → `Dia5-Create-Tables-Indian` → `Dia6b-Fato-Vendas` |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## DataFrame API — Filtro e agregação

Carrega `fat_vendas` como DataFrame, filtra vendas com valor > 0 e agrega receita diária.

In [0]:
from pyspark.sql import functions as F

df1 = spark.table(f'{catalog}.gold.fat_vendas')

result = (df1
  .filter(F.col('ven_vl_total_pago') > 0)
  .groupBy('ven_sk_data_venda')
  .agg(F.sum('ven_vl_total_pago').alias('daily_revenue')))

result.show()

## Lazy evaluation

Demonstra que o Spark só executa transformações quando uma action é chamada (`.write`).

In [0]:
df2 = (spark.table(f'{catalog}.bronze.sales')
  .filter(F.col('Order_Date') == '2026-09-01')
  .select('Order_ID','Customer_ID','Order_Value')
  .groupBy('Customer_ID')
  .agg(F.sum(F.col('Order_Value')).alias('total_revenue')))

df2.show()

## Leitura, filtro, join e enriquecimento

Carrega as tabelas Bronze, aplica filtros, cria coluna calculada (`line_total`) e faz `LEFT JOIN` com customers e products para enriquecer as vendas.

In [0]:
sales = spark.read.table(f'{catalog}.bronze.sales')
customers = spark.read.table(f'{catalog}.bronze.customers')
products = spark.read.table(f'{catalog}.bronze.products')

sales_clean = (sales
  .filter(F.col('Shipping_Cost') > 0)
  .filter(F.col('Order_Date').isNotNull())
  .select('Order_ID','Customer_ID','Product_ID','Order_Date','Quantity','Unit_Price')
  .withColumn('line_total', F.col('Quantity') * F.col('Unit_Price'))
)

sales_enriched = (sales_clean
  .join(customers, on='Customer_ID', how='left')
  .join(products, on='Product_ID', how='left')
)

## Agregação por data e categoria

Agrupa vendas enriquecidas por data e categoria, calculando receita, pedidos e unidades.

In [0]:
daily_revenue = (sales_enriched
  .groupBy('order_date','category')
  .agg(
    F.sum('line_total').alias('revenue'),
    F.countDistinct('Order_ID').alias('orders'),
    F.sum('Quantity').alias('units')
  )
  .orderBy('order_date', F.desc('revenue'))
)

daily_revenue.show()

## Escrita em Delta

Salva o resultado da agregação na tabela Gold `daily_revenue` com `overwrite` (idempotente).

In [0]:
(daily_revenue.write
  .format('delta')
  .mode('overwrite')
  .option('overwriteSchema', 'true')
  .saveAsTable(f'{catalog}.gold.daily_revenue')
)

## Inspeção do plano de execução

Usa `explain()` para visualizar o plano físico que o Spark montou — útil para diagnóstico de performance.

In [0]:
daily_revenue.explain(mode='formatted')

## Join avançado com dimensões Gold

Faz `LEFT JOIN` da fato `fat_vendas` com `dim_produtos` e `dim_datas` via chaves substitutas (SK), e agrega receita por ano, mês e categoria.

In [0]:
schema = 'gold'
source_table = f'{catalog}.{schema}.fat_vendas'
product_table = f'{catalog}.{schema}.dim_produtos'
date_table = f'{catalog}.{schema}.dim_datas'
target_table = f'{catalog}.{schema}.revenue_by_category'

fato = spark.table(source_table)
produtos = spark.table(product_table)
datas = spark.table(date_table)

result = (fato
  .join(produtos, on=[fato['ven_sk_produto'] == produtos['pro_sk_produto']], how='left')
  .join(datas, on=[fato['ven_sk_data_venda'] == datas['dat_sk_data']], how='left')
  .groupBy('dat_nu_ano','dat_nu_mes','pro_ds_categoria')
  .agg(
    F.sum('ven_vl_total_pago').alias('revenue'),
    F.countDistinct('ven_id_venda').alias('orders'),
    F.sum('ven_qt_itens').alias('units')
  )
  .withColumnRenamed('dat_nu_ano', 'ano')
  .withColumnRenamed('dat_nu_mes', 'mes')
  .withColumnRenamed('pro_ds_categoria', 'categoria')
  .orderBy('ano', 'mes', F.desc('revenue'))
)

In [0]:
(result.write
  .format('delta')
  .mode('overwrite')
  .option('overwriteSchema','true')
  .saveAsTable(target_table))

print("Total de linhas:", spark.table(target_table).count())
print("Linhas com revenue nulo:", spark.table(target_table).filter(F.col('revenue').isNull()).count())

## Reset opcional

Se precisar reiniciar o laboratório, mude `RESET_LAB = True` e execute para apagar as tabelas de teste.

In [0]:
RESET_LAB = False

if RESET_LAB:
    for t in [f'{catalog}.silver.customer_revenue', f'{catalog}.gold.daily_revenue', target_table]:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    print("Laboratorio reiniciado.")
else:
    print("RESET_LAB = False. Nenhum objeto foi removido.")

In [0]:
# explain() mostra o "plano de execução" que o Spark montou por trás dos panos — como ele vai
# ler, filtrar, juntar e agregar os dados fisicamente. Útil pra investigar performance depois.
daily_revenue.explain(mode='formatted')

daily_revenue.explain(mode='extended')

== Physical Plan ==
AdaptiveSparkPlan (27)
+- == Initial Plan ==
   PhotonResultStage (26)
   +- PhotonColumnarToRow (25)
      +- PhotonGroupingAgg (24)
         +- PhotonShuffleExchangeSource (23)
            +- PhotonShuffleMapStage (22)
               +- PhotonShuffleExchangeSink (21)
                  +- PhotonGroupingAgg (20)
                     +- PhotonGroupingAgg (19)
                        +- PhotonShuffleExchangeSource (18)
                           +- PhotonShuffleMapStage (17)
                              +- PhotonShuffleExchangeSink (16)
                                 +- PhotonGroupingAgg (15)
                                    +- PhotonProject (14)
                                       +- PhotonBroadcastHashJoin LeftOuter (13)
                                          :- PhotonProject (8)
                                          :  +- PhotonBroadcastHashJoin LeftOuter (7)
                                          :     :- PhotonProject (2)
                      

In [0]:
from pyspark.sql import functions as F

schema = 'gold'
source_table = f'{catalog}.{schema}.fat_vendas'
product_table = f'{catalog}.{schema}.dim_produtos'
date_table = f'{catalog}.{schema}.dim_datas'
target_table = f'{catalog}.{schema}.revenue_by_month_category_pyspark'

fato = spark.table(source_table)
produtos = spark.table(product_table)
datas = spark.table(date_table)